In [1]:
import json
import pandas as pd
from collections import Counter
from sklearn.metrics import cohen_kappa_score, f1_score

In [3]:
file_path = "batch1_20docs_export.json"

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Total tasks in file:", len(data))

Total tasks in file: 20


In [5]:
overlap_tasks = [task for task in data if len(task.get("annotations", [])) >= 2]

print("Overlap tasks (>=2 annotations):", len(overlap_tasks))

Overlap tasks (>=2 annotations): 20


In [7]:
annotators = set()

for task in overlap_tasks:
    for ann in task["annotations"][:2]:
        email = ann.get("completed_by", {}).get("email", "unknown")
        annotators.add(email)

print("Annotators found:")
for a in annotators:
    print("-", a)

Annotators found:
- ysasson@lion.lmu.edu
- bacosta3@lion.lmu.edu


In [9]:
def tokenize_with_spans(text):
    tokens = []
    start_search = 0
    
    for token in text.split():
        start = text.find(token, start_search)
        end = start + len(token)
        tokens.append((token, start, end))
        start_search = end
    
    return tokens

In [11]:
def annotation_to_token_labels(text, annotation_result):
    tokens = tokenize_with_spans(text)

    binary_labels = [0] * len(tokens)
    exact_labels = ["O"] * len(tokens)

    for res in annotation_result:
        if res.get("type") != "labels":
            continue

        value = res.get("value", {})
        start_char = value.get("start")
        end_char = value.get("end")
        labels = value.get("labels", [])

        if start_char is None or end_char is None or not labels:
            continue

        entity_label = labels[0]

        matched = []
        for i, (_, tok_s, tok_e) in enumerate(tokens):
            if tok_s < end_char and tok_e > start_char:
                matched.append(i)

        for j, idx in enumerate(matched):
            binary_labels[idx] = 1
            exact_labels[idx] = f"B-{entity_label}" if j == 0 else f"I-{entity_label}"

    token_texts = [t[0] for t in tokens]
    return token_texts, binary_labels, exact_labels

In [13]:
paired_tasks = []

for task in overlap_tasks:
    text = task["data"]["text"]

    ann_a = task["annotations"][0]
    ann_b = task["annotations"][1]

    email_a = ann_a.get("completed_by", {}).get("email", "annotator_a")
    email_b = ann_b.get("completed_by", {}).get("email", "annotator_b")

    tokens_a, binary_a, exact_a = annotation_to_token_labels(text, ann_a.get("result", []))
    tokens_b, binary_b, exact_b = annotation_to_token_labels(text, ann_b.get("result", []))

    if tokens_a == tokens_b:
        paired_tasks.append({
            "text": text,
            "annotator_a": email_a,
            "annotator_b": email_b,
            "tokens": tokens_a,
            "binary_a": binary_a,
            "binary_b": binary_b,
            "exact_a": exact_a,
            "exact_b": exact_b
        })
    else:
        print("Token mismatch for text:", text)

print("Valid paired tasks:", len(paired_tasks))

Valid paired tasks: 20


In [15]:
all_binary_a, all_binary_b = [], []
all_exact_a, all_exact_b = [], []

for t in paired_tasks:
    all_binary_a.extend(t["binary_a"])
    all_binary_b.extend(t["binary_b"])
    all_exact_a.extend(t["exact_a"])
    all_exact_b.extend(t["exact_b"])

print("Total tokens compared:", len(all_binary_a))

Total tokens compared: 151


In [17]:
kappa_binary = cohen_kappa_score(all_binary_a, all_binary_b)
f1_binary = f1_score(all_binary_a, all_binary_b)

print("Binary token-level (entity vs O):")
print(f"  Cohen's Kappa : {kappa_binary:.4f}")
print(f"  F1 Score      : {f1_binary:.4f}")

Binary token-level (entity vs O):
  Cohen's Kappa : 0.9712
  F1 Score      : 0.9815


In [19]:
exact_matches = sum(1 for a, b in zip(all_exact_a, all_exact_b) if a == b)
raw_agreement = exact_matches / len(all_exact_a)

kappa_exact = cohen_kappa_score(all_exact_a, all_exact_b)

print(f"Raw exact agreement      : {raw_agreement*100:.2f}%")
print(f"Cohen's Kappa (B/I tags) : {kappa_exact:.4f}")

Raw exact agreement      : 96.69%
Cohen's Kappa (B/I tags) : 0.9422


In [21]:
disagreements = []

for t in paired_tasks:
    for tok, a, b in zip(t["tokens"], t["exact_a"], t["exact_b"]):
        if a != b:
            disagreements.append({
                "text": t["text"],
                "token": tok,
                "annotator_a": a,
                "annotator_b": b
            })

disagreement_df = pd.DataFrame(disagreements)

print("Total token disagreements:", len(disagreement_df))
disagreement_df.head(25)

Total token disagreements: 5


,text,token,annotator_a,annotator_b
0,can you find me the nearest pizzeria,pizzeria,B-Dish,B-Cuisine
1,check for cheap vegan diners in the theater di...,vegan,B-Cuisine,B-Amenity
2,check for cheap vegan diners in the theater di...,in,B-Location,O
3,check for cheap vegan diners in the theater di...,the,I-Location,O
4,check for cheap vegan diners in the theater di...,theater,I-Location,B-Location


In [23]:
def collapse_bio(tag):
    return tag[2:] if tag.startswith(("B-", "I-")) else tag

type_conflicts = Counter()

for row in disagreements:
    ta = collapse_bio(row["annotator_a"])
    tb = collapse_bio(row["annotator_b"])
    
    pair = tuple(sorted([ta, tb]))
    type_conflicts[pair] += 1

conflict_df = (
    pd.DataFrame(
        [(a, b, n) for (a, b), n in type_conflicts.items()],
        columns=["label_a", "label_b", "count"]
    )
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

print("Disagreement pairs by entity type (collapsed B/I):")
conflict_df

Disagreement pairs by entity type (collapsed B/I):


,label_a,label_b,count
0,Location,O,2
1,Cuisine,Dish,1
2,Amenity,Cuisine,1
3,Location,Location,1


In [25]:
print("Results summary")
print(f"Overlap sample size: {len(paired_tasks)} tasks")
print(f"Binary Cohen's Kappa: {kappa_binary:.4f}")
print(f"Binary F1: {f1_binary:.4f}")
print(f"Raw exact agreement: {raw_agreement*100:.2f}%")
print(f"BIO-tag Cohen's Kappa: {kappa_exact:.4f}")

Results summary
Overlap sample size: 20 tasks
Binary Cohen's Kappa: 0.9712
Binary F1: 0.9815
Raw exact agreement: 96.69%
BIO-tag Cohen's Kappa: 0.9422
